# Autograd : distinguer poids, perte et dérivée

Mesurer une influence locale avant de demander à un optimiseur de déplacer un poids.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#autograd-et-derivees).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Nous quittons un instant la classification pour un calcul scalaire entièrement vérifiable. Le mécanisme de dérivation sera ensuite réutilisé sur sa perte.

Choisissons L=(w−2)² et w=0. La perte vaut 4 ; le minimum vaut zéro lorsque w atteint 2. La dérivée 2(w−2) vaut −4 au point de départ.

requires_grad=True demande le suivi de w. La notation 0. signifie 0.0 en Python : un nombre flottant nécessaire à ce calcul différentiable.

backward remonte les opérations depuis la perte. w.grad contient ensuite un tenseur de dérivée ; w garde sa valeur. item() extrait un seul nombre pour l’affichage.

## Démonstration 1 · Observer la pente et une petite variation

Ne pas confondre le paramètre, son coût et sa dérivée.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

w est un tenseur de forme (), avec requires_grad=True. La soustraction puis le carré produisent la perte.

In [4]:
w = torch.tensor(0., requires_grad=True)
perte = (w - 2.) ** 2


In [5]:
observer('w', w)
observer('perte', perte)

w forme () type torch.float32 requires_grad True
  valeurs : 0.0
perte forme () type torch.float32 requires_grad True
  valeurs : 4.0


### Calculer les gradients

backward remplit w.grad. Il n’exécute aucune règle de mise à jour du paramètre.

In [6]:
perte.backward()


In [7]:
observer_entrainement({'w': w, 'perte': perte}, montrer_poids=False)

w.grad forme () type torch.float32 requires_grad False
  valeurs : -4.0


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [8]:
print("poids", w.item(), "perte", perte.item(), "gradient", w.grad.item())


poids 0.0 perte 4.0 gradient -4.0


### Calculer l’état suivant

Nous évaluons la formule en 0 puis 0,01. La variation divisée par 0,01 approche la dérivée locale.

In [9]:
avant = (0. - 2.) ** 2
apres = (0.01 - 2.) ** 2
pente_approchee = (apres - avant) / 0.01


In [10]:
observer('avant', avant)
observer('apres', apres)
observer('pente_approchee', pente_approchee)

avant = 4.0
apres = 3.9601
pente_approchee = -3.9899999999999824


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [11]:
print("petite variation", round(apres, 4), "pente approchee", round(pente_approchee, 2))


petite variation 3.9601 pente approchee -3.99


### Calculer l’état suivant

u est une expérience indépendante. Deux calculs neufs 3u puis 4u ajoutent leurs contributions dans le même champ grad.

In [12]:
u = torch.tensor(1., requires_grad=True)


In [13]:
observer('u', u)

u forme () type torch.float32 requires_grad True
  valeurs : 1.0


### Calculer les gradients

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [14]:
(3 * u).backward()


In [15]:
observer_entrainement({'w': w, 'perte': perte, 'avant': avant, 'apres': apres, 'pente_approchee': pente_approchee, 'u': u}, montrer_poids=False)

w.grad forme () type torch.float32 requires_grad False
  valeurs : -4.0
u.grad forme () type torch.float32 requires_grad False
  valeurs : 3.0


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [16]:
print("premier gradient", u.grad.item())


premier gradient 3.0


### Calculer les gradients

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [17]:
(4 * u).backward()


In [18]:
observer_entrainement({'w': w, 'perte': perte, 'avant': avant, 'apres': apres, 'pente_approchee': pente_approchee, 'u': u}, montrer_poids=False)

w.grad forme () type torch.float32 requires_grad False
  valeurs : -4.0
u.grad forme () type torch.float32 requires_grad False
  valeurs : 7.0


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [19]:
print("gradient cumule", u.grad.item())

gradient cumule 7.0


Le signe négatif signifie qu’une petite augmentation de w réduit la perte à cet endroit. Il ne signifie pas que la perte elle-même est négative.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Prévoyez le gradient de (w−2)² au point w=0.

In [20]:
exo_w = torch.tensor(0.,requires_grad=True)
exo_perte = (exo_w-2.)**2
exo_perte.backward()
exo_gradient = ...  # Remplacer les points par votre réponse.
if exo_gradient is Ellipsis or exo_gradient is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_gradient)-exo_w.grad.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

La dérivée vaut 2×(w−2), évaluée au poids actuel.

</details>

### Exercice 2

Deux backward successifs sur 3u puis 4u, sans remise à zéro : que contient u.grad ?

In [21]:
exo_u = torch.tensor(1.,requires_grad=True)
(3*exo_u).backward()
(4*exo_u).backward()
exo_gradient = ...  # Remplacer les points par votre réponse.
if exo_gradient is Ellipsis or exo_gradient is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_gradient)-exo_u.grad.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Les contributions des deux graphes s’accumulent dans le même champ grad.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_w = torch.tensor(0.,requires_grad=True)
exo_perte = (exo_w-2.)**2
exo_perte.backward()
exo_gradient = -4.
if exo_gradient is Ellipsis or exo_gradient is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_gradient)-exo_w.grad.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_u = torch.tensor(1.,requires_grad=True)
(3*exo_u).backward()
(4*exo_u).backward()
exo_gradient = 7.
if exo_gradient is Ellipsis or exo_gradient is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_gradient)-exo_u.grad.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [22]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()